# Posterior Shift Simulator
## Module 1 — The Bayesian Frame: Prompts as Evidence

### Learning Objectives

By completing this notebook, you will:
1. Observe directly how adding conditions to a prompt changes Claude's response
2. Measure response specificity as a proxy for posterior narrowing
3. Identify the stabilization point where additional conditions stop changing the response
4. Build intuition for which conditions shift the posterior in different domains

### Prerequisites
- An Anthropic API key set as `ANTHROPIC_API_KEY` in your environment
- Guide 01 (Prompts as Evidence) read

### Estimated Time: 15 minutes

---

**The Core Claim We Are Testing**

Guide 01 claims that adding discriminating conditions shifts the model's response away from its training prior toward a more specific answer — and that this effect stabilizes once enough conditions are provided. Let's test this claim directly.

In [ ]:
learning_objectives(["— The Bayesian Frame: Prompts as Evidence", "Observe directly how adding conditions to a prompt changes Claude's response", "Measure response specificity as a proxy for posterior narrowing", "Identify the stabilization point where additional conditions stop changing the response", "Build intuition for which conditions shift the posterior in different domains", "An Anthropic API key set as `ANTHROPIC_API_KEY` in your environment"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import anthropic
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from difflib import SequenceMatcher
import textwrap
import os

# Initialize client — reads ANTHROPIC_API_KEY from environment
client = anthropic.Anthropic()

print("Client initialized. Model: claude-opus-4-5")

## 1. The Condition Ladder

We'll test three domains. For each domain, we build a **condition ladder**: a base question with progressively more discriminating conditions added one at a time.

The prediction from Guide 01:
- Early conditions should produce the largest response changes (posterior shifts dramatically away from prior)
- Later conditions should produce smaller changes (posterior is already narrowed, stabilizing)
- Some conditions should produce **no visible change** (they were information, not evidence)

We measure response change using text similarity: higher similarity = smaller posterior shift from the previous step.

In [ ]:
# Three domains: tax, medical, code debugging
# Each domain has a base question plus a sequence of conditions to add one at a time

domains = {
    "Tax": {
        "base": "What should I know about estimated tax payments?",
        "conditions": [
            # Step 1: Adds entity type — discriminating (shifts away from individual 1040)
            "I am self-employed as a sole proprietor.",
            # Step 2: Adds income threshold — discriminating (triggers specific SE tax rules)
            "My net self-employment income is approximately $85,000 this year.",
            # Step 3: Adds a constraint — discriminating (rules out W-2 withholding offset)
            "I have no W-2 income and no other withholding.",
            # Step 4: Adds timing — discriminating (specific quarter deadline applies)
            "I missed the Q2 estimated payment and it is now August.",
            # Step 5: Information only — should NOT shift posterior
            "I want to make sure I understand this clearly.",
        ],
        "condition_labels": [
            "Base only",
            "+ Entity type",
            "+ Income level",
            "+ No withholding",
            "+ Missed Q2",
            "+ 'Clearly' (filler)",
        ]
    },
    "Code": {
        "base": "My code isn't working as expected. How do I debug it?",
        "conditions": [
            # Step 1: Adds language — discriminating (debugging tools are language-specific)
            "I am working in Python.",
            # Step 2: Adds failure mode — discriminating (rules out syntax/import errors)
            "The script runs without errors but produces wrong output.",
            # Step 3: Adds threshold — discriminating (suggests numerical or scaling issue)
            "The output is correct for small inputs but wrong when the list has more than 1000 items.",
            # Step 4: Adds constraint — discriminating (rules out external profiling tools)
            "I cannot install any additional packages.",
            # Step 5: Information only — should NOT shift posterior
            "I've been working on this for a while and I'm quite frustrated.",
        ],
        "condition_labels": [
            "Base only",
            "+ Language: Python",
            "+ Runs, wrong output",
            "+ Threshold at 1000",
            "+ No extra packages",
            "+ 'Frustrated' (filler)",
        ]
    },
    "Strategy": {
        "base": "How should we approach customer retention?",
        "conditions": [
            # Step 1: Adds business type — discriminating (B2B vs B2C very different)
            "We are a B2B SaaS company.",
            # Step 2: Adds metric — discriminating (reveals where the problem is)
            "Our monthly churn rate is 8%, which is above the industry average of 5%.",
            # Step 3: Adds failure mode — discriminating (rules out acquisition-stage solutions)
            "Exit surveys show customers leave because they never achieved their initial goal.",
            # Step 4: Adds constraint — discriminating (rules out product changes)
            "We cannot change the core product features in the next quarter.",
            # Step 5: Information only — should NOT shift posterior
            "We are very motivated to solve this problem.",
        ],
        "condition_labels": [
            "Base only",
            "+ B2B SaaS",
            "+ 8% churn (above avg)",
            "+ Exit survey: goal failure",
            "+ No product changes",
            "+ 'Motivated' (filler)",
        ]
    }
}

print("Domains defined. Each has 5 conditions to add progressively.")
print("Conditions 1-4 are designed to be discriminating evidence.")
print("Condition 5 is a filler (information, not evidence) — it should NOT shift the posterior.")

## 2. Query Function and Similarity Metric

We send each step's prompt to Claude and record the response. To measure how much each condition shifted the posterior, we compare adjacent responses using sequence similarity.

**Similarity score interpretation:**
- Score near 1.0: responses are nearly identical — condition did not shift posterior
- Score near 0.0: responses are completely different — condition shifted posterior dramatically

We expect: high dissimilarity for early discriminating conditions, low dissimilarity for the filler condition.

In [ ]:
def build_prompt(base: str, conditions: list[str], num_conditions: int) -> str:
    """Build a prompt by combining a base question with N conditions.
    
    Conditions are prepended to the base question to act as context.
    This mirrors real usage where background information precedes the question.
    
    Args:
        base: The core question without conditions
        conditions: List of condition strings to prepend
        num_conditions: How many conditions to include (0 = base only)
    
    Returns:
        Complete prompt string
    """
    if num_conditions == 0:
        return base
    # Build context from the first N conditions
    context = " ".join(conditions[:num_conditions])
    return f"{context} {base}"


def get_response(prompt: str, max_tokens: int = 400) -> str:
    """Send a prompt to Claude and return the text response.
    
    Using claude-opus-4-5 for consistent, high-quality responses.
    max_tokens capped to keep responses comparable across steps.
    """
    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=max_tokens,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text


def response_similarity(text_a: str, text_b: str) -> float:
    """Compute normalized similarity between two response strings.
    
    Uses SequenceMatcher for character-level comparison.
    Returns 0.0 (completely different) to 1.0 (identical).
    
    Lower similarity = larger posterior shift between these two steps.
    """
    return SequenceMatcher(None, text_a.lower(), text_b.lower()).ratio()


def specificity_score(text: str) -> float:
    """Estimate response specificity as a proxy for posterior narrowness.
    
    Heuristic: specific responses tend to be longer, use more concrete terms,
    and contain fewer hedge words. We use a simple hedge-word ratio.
    
    Lower hedge ratio = more specific = narrower posterior.
    
    Returns a value in [0, 1] where higher = more specific.
    """
    hedge_words = [
        "generally", "typically", "usually", "often", "in most cases",
        "it depends", "may vary", "consider", "might", "could",
        "various", "different", "multiple options", "several", "some"
    ]
    text_lower = text.lower()
    hedge_count = sum(text_lower.count(hw) for hw in hedge_words)
    # Normalize by word count; invert so higher = more specific
    word_count = max(len(text.split()), 1)
    hedge_ratio = hedge_count / word_count
    # Clamp and invert
    return max(0.0, 1.0 - min(hedge_ratio * 10, 1.0))


print("Functions defined.")
print("build_prompt(): constructs prompt at each step of the condition ladder")
print("get_response(): calls Claude API with capped tokens for comparability")
print("response_similarity(): measures how much adjacent responses differ")
print("specificity_score(): heuristic measure of response specificity via hedge words")

## 3. Run the Condition Ladder

For each domain, we send 6 prompts (base + 5 condition steps) to Claude. This takes approximately 30-60 seconds per domain due to API calls.

The results are stored for visualization.

In [ ]:
# Run all domains and collect responses
# Each domain: 6 API calls (base + 5 conditions)
# Total: 18 API calls

results = {}

for domain_name, domain_data in domains.items():
    print(f"\nRunning domain: {domain_name}")
    print("-" * 40)
    
    responses = []
    prompts_sent = []
    
    for step in range(6):  # 0 = base only, 1-5 = with N conditions
        prompt = build_prompt(
            domain_data["base"],
            domain_data["conditions"],
            num_conditions=step
        )
        prompts_sent.append(prompt)
        
        print(f"  Step {step} ({domain_data['condition_labels'][step]}): ", end="", flush=True)
        response = get_response(prompt)
        responses.append(response)
        print(f"{len(response)} chars")
    
    # Compute similarity between each adjacent pair of responses
    # similarity[i] = how similar step i is to step i-1
    similarities = [None]  # No predecessor for step 0
    for i in range(1, len(responses)):
        sim = response_similarity(responses[i-1], responses[i])
        similarities.append(sim)
    
    # Compute specificity at each step
    specificities = [specificity_score(r) for r in responses]
    
    results[domain_name] = {
        "responses": responses,
        "prompts": prompts_sent,
        "similarities": similarities,
        "specificities": specificities,
        "labels": domain_data["condition_labels"]
    }

print("\nAll domains complete.")

## 4. Visualize the Posterior Shift

Two visualizations:

1. **Similarity to previous step** — Lower bars = larger shift. We expect large shifts at discriminating conditions and small shifts at the filler condition.
2. **Specificity over time** — Should increase as conditions are added and stabilize.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(
    "Posterior Shift Simulator\nHow Adding Conditions Changes Claude's Response",
    fontsize=14, fontweight="bold", y=1.02
)

domain_names = list(results.keys())
colors_shift = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#95a5a6"]  # red=large shift, gray=no shift

for col, domain_name in enumerate(domain_names):
    data = results[domain_name]
    steps = list(range(1, 6))  # Steps 1-5 have similarity scores
    sims = [data["similarities"][i] for i in steps]
    specs = data["specificities"]
    labels_short = [data["labels"][i].replace("+ ", "") for i in steps]
    
    # --- Top row: similarity to previous step (inverted = shift magnitude) ---
    ax_top = axes[0, col]
    shift_magnitudes = [1.0 - s for s in sims]  # Invert: high bar = large shift
    
    bar_colors = []
    for i, label in enumerate(data["labels"][1:]):
        if "filler" in label.lower() or "clearly" in label.lower() or "frustrated" in label.lower() or "motivated" in label.lower():
            bar_colors.append("#95a5a6")  # Gray for filler conditions
        else:
            bar_colors.append(colors_shift[i % len(colors_shift)])
    
    bars = ax_top.bar(range(5), shift_magnitudes, color=bar_colors, edgecolor="white", linewidth=0.5)
    ax_top.set_xticks(range(5))
    ax_top.set_xticklabels(labels_short, rotation=30, ha="right", fontsize=7.5)
    ax_top.set_ylabel("Shift Magnitude\n(1 - similarity)", fontsize=9)
    ax_top.set_title(f"{domain_name}\nPosterior Shift at Each Step", fontsize=10, fontweight="bold")
    ax_top.set_ylim(0, 1.0)
    ax_top.axhline(y=0.15, color="navy", linestyle="--", linewidth=0.8, alpha=0.5, label="Threshold: meaningful shift")
    ax_top.legend(fontsize=7)
    
    # Annotate bars with similarity values
    for i, (bar, sim) in enumerate(zip(bars, sims)):
        ax_top.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{sim:.2f}",
            ha="center", va="bottom", fontsize=7.5, color="#2c3e50"
        )
    
    # --- Bottom row: specificity over time ---
    ax_bot = axes[1, col]
    ax_bot.plot(
        range(6), specs,
        marker="o", markersize=7,
        color="#4a90d9", linewidth=2, label="Specificity"
    )
    ax_bot.fill_between(range(6), specs, alpha=0.15, color="#4a90d9")
    ax_bot.set_xticks(range(6))
    all_labels_short = [data["labels"][i].replace("+ ", "").replace("Base only", "Base") for i in range(6)]
    ax_bot.set_xticklabels(all_labels_short, rotation=30, ha="right", fontsize=7.5)
    ax_bot.set_ylabel("Specificity Score\n(1 - hedge ratio)", fontsize=9)
    ax_bot.set_title(f"Response Specificity Over Steps", fontsize=10)
    ax_bot.set_ylim(0, 1.1)
    
    # Mark the filler step (step 5, index 5)
    ax_bot.axvline(x=5, color="#95a5a6", linestyle=":", linewidth=1.5, label="Filler condition")
    ax_bot.legend(fontsize=7)

# Legend for shift bar colors
legend_patches = [
    mpatches.Patch(color="#e74c3c", label="Discriminating evidence"),
    mpatches.Patch(color="#95a5a6", label="Filler information (no shift expected)"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig("../resources/posterior_shift_visualization.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved to resources/posterior_shift_visualization.png")

## 5. Read the Responses: Before and After

Numbers tell part of the story. Let's read the actual responses to see the qualitative shift. We'll compare Step 0 (base only) against Step 4 (all discriminating conditions added) for each domain.

In [ ]:
def print_comparison(domain_name: str, data: dict, step_a: int = 0, step_b: int = 4):
    """Print a side-by-side comparison of two response steps for a domain."""
    print("=" * 70)
    print(f"DOMAIN: {domain_name}")
    print("=" * 70)
    
    print(f"\nPROMPT at Step {step_a} ({data['labels'][step_a]}):")
    print("-" * 50)
    print(data["prompts"][step_a])
    
    print(f"\nRESPONSE at Step {step_a}:")
    print("-" * 50)
    # Wrap for readability
    for line in data["responses"][step_a].split("\n"):
        print(textwrap.fill(line, width=70) if line else "")
    
    print(f"\n\nPROMPT at Step {step_b} ({data['labels'][step_b]}):")
    print("-" * 50)
    print(data["prompts"][step_b])
    
    print(f"\nRESPONSE at Step {step_b}:")
    print("-" * 50)
    for line in data["responses"][step_b].split("\n"):
        print(textwrap.fill(line, width=70) if line else "")
    
    sim = response_similarity(data["responses"][step_a], data["responses"][step_b])
    print(f"\nSimilarity between Step {step_a} and Step {step_b}: {sim:.3f}")
    print(f"Specificity at Step {step_a}: {data['specificities'][step_a]:.3f}")
    print(f"Specificity at Step {step_b}: {data['specificities'][step_b]:.3f}")


# Show the Tax domain comparison — most concrete for this module's examples
print_comparison("Tax", results["Tax"])

In [ ]:
# Show the Code domain comparison
print_comparison("Code", results["Code"])

## 6. The Filler Test

Step 5 in each domain adds a filler sentence — information that is true but should not discriminate between possible answers. Guide 02 predicts these sentences will not shift the posterior.

Let's check: how similar is Step 5's response to Step 4's response?

In [ ]:
print("Filler Condition Test\n" + "=" * 50)
print("Prediction: filler conditions produce HIGH similarity (low shift)")
print("Discriminating conditions produce LOW similarity (high shift)\n")

for domain_name, data in results.items():
    print(f"Domain: {domain_name}")
    print(f"  Filler condition: '{domains[domain_name]['conditions'][-1]}'")
    
    # Similarity between step 4 (all real conditions) and step 5 (+ filler)
    filler_sim = data["similarities"][5]
    
    # Average similarity of discriminating conditions (steps 1-4)
    discriminating_sims = [data["similarities"][i] for i in range(1, 5)]
    avg_discriminating_sim = np.mean(discriminating_sims)
    
    print(f"  Similarity after filler: {filler_sim:.3f} (prediction: high, > 0.70)")
    print(f"  Avg similarity after discriminating conditions: {avg_discriminating_sim:.3f} (prediction: lower)")
    
    result = "CONFIRMED" if filler_sim > avg_discriminating_sim else "CHECK"
    print(f"  Filler shifts less than discriminating conditions: {result}")
    print()

## 7. Exercise: Your Own Condition Ladder

Choose a domain from your own work. Build a condition ladder with:
- 1 base question
- 4 discriminating conditions (use the four categories from Guide 02: constraint, timing/jurisdiction, objective function, failure mode)
- 1 filler sentence

Predict the similarity and specificity curves before running. Then run and compare.

In [ ]:
# Exercise: Define your own domain
# Replace the placeholders below with your actual content
# ---------------------------------------------------------------------------

my_domain = {
    "base": "[YOUR BASE QUESTION HERE]",
    "conditions": [
        "[DISCRIMINATING CONDITION 1: constraint]",
        "[DISCRIMINATING CONDITION 2: timing or jurisdiction]",
        "[DISCRIMINATING CONDITION 3: objective function]",
        "[DISCRIMINATING CONDITION 4: failure mode or specific fact]",
        "[FILLER CONDITION: true but should not change the answer]",
    ],
    "condition_labels": [
        "Base only",
        "+ Constraint",
        "+ Timing/Jurisdiction",
        "+ Objective",
        "+ Failure mode",
        "+ Filler",
    ]
}

# ---------------------------------------------------------------------------
# Uncomment and run after filling in the above

# my_responses = []
# for step in range(6):
#     prompt = build_prompt(my_domain["base"], my_domain["conditions"], step)
#     response = get_response(prompt)
#     my_responses.append(response)
#     print(f"Step {step}: {len(response)} chars")
#
# my_similarities = [None] + [
#     response_similarity(my_responses[i-1], my_responses[i])
#     for i in range(1, 6)
# ]
# my_specificities = [specificity_score(r) for r in my_responses]
#
# for i, label in enumerate(my_domain["condition_labels"]):
#     sim_str = f", similarity from prev: {my_similarities[i]:.3f}" if i > 0 else ""
#     print(f"{label}: specificity={my_specificities[i]:.3f}{sim_str}")

print("Exercise cell ready. Fill in my_domain above and uncomment the run block.")

## Summary

### What We Observed

1. **Discriminating conditions shift the posterior** — response similarity drops significantly when relevant conditions are added
2. **Filler conditions do not** — sentences like "I want to understand this clearly" produce high similarity, confirming they don't shift the posterior
3. **Specificity increases with conditions** — hedge words and general caveats decrease as the posterior narrows
4. **Stabilization is real** — later conditions produce smaller shifts than earlier ones; the posterior converges

### The Formula, Verified

$$P(A \mid C) \propto P(C \mid A) \cdot P(A)$$

| Observation | Formula Explanation |
|-------------|--------------------|
| Base prompt → generic answer | C is weak; P(A) dominates |
| Discriminating condition → shifted answer | C narrows P(C|A) for wrong answers toward 0 |
| Filler condition → same answer | C doesn't change P(C|A) ratio |
| Later conditions → smaller shifts | Posterior already narrowed; diminishing returns |

### What's Next

Notebook 02 takes 5 real prompts through the full diagnostic: weak → detail-added (still generic) → key condition (specific). The goal is to build the muscle memory of identifying the missing key condition.
